In [15]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os

In [16]:
# ── PATHS ────────────────────────────────────────────────────────────────────────────────────────────

# Root folder holding all subfolders containing raw downloaded GEBCO .tif files for each section
root = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Raw_data\GEBCO_raw_tif_downloads_g4326"

# Output geodatabase for output GEBCO mosaic
gdb = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb"

# Output standalone raster created from mosaic
final_raster = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_withland_g4326"

# Output footprint polygon
md_footprint = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\footprint_RAW_GEBCO_MOSAIC_g4326"

In [17]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun
arcpy.env.compression = "NONE"      # set compression to NONE for output mosaic and raster

# set environment coordinate system: horizontal EPSG:4326 *TO MATCH NATIVE GEBCO COORDINATE SYSTEM OF INPUT TIFF FILES* (coord system set within Create Mosaic Dataset tool *should* be carried over by Copy Raster tool from the input mosaic, but set explicitly for entire environment just to be safe)
arcpy.env.outputCoordinateSystem = arcpy.SpatialReference(4326)

In [18]:
# ── 1. CREATE NEW MOSAIC DATASET ─────────────────────────────────────────────────────────────────────

# Create Mosaic Dataset (Data Management Tools):
try:
    arcpy.management.CreateMosaicDataset(
        in_workspace=gdb,
        in_mosaicdataset_name="RAW_GEBCO_MOSAIC_g4326",
        coordinate_system=arcpy.SpatialReference(4326)
        # num_bands leave blank so defaults based on inputs when rasters are added (1 band for downloaded GEBCO .tif files)
        # pixel_type defaults to bit depth of first input raster (16-bit short for all downloaded GEBCO .tif files)
        # product_definition defaults to "NONE" (generic)
    )
    # define path to new mosaic for use in future blocks
    md = os.path.join(gdb, "RAW_GEBCO_MOSAIC_g4326")
    print(f"✅ Created mosaic dataset: {md} \n")

# print warning if tool failed
except Exception as e:
    print(f"❌[FAILURE] Mosaic creation failed: {e} \n")

✅ Created mosaic dataset: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\RAW_GEBCO_MOSAIC_g4326 



In [19]:
# ── SELECT ONLY TIFFS WITH BATHYMETRY DATA FOR INPUT INTO MOSAIC ─────────────────────────────────────

# Create tifs[] list to store paths of ONLY depth grids .tif files (actual bathymetry data) in each subfolder, skip TID (type identifier) .tif files
tifs = []
for dirpath, _, files in os.walk(root):
    for f in files:
        if f.lower().endswith(".tif") and "_tid_" not in f.lower():
            tifs.append(os.path.join(dirpath, f))

In [20]:
# ── 2. ADD GEBCO TIFF RASTERS TO MOSAIC DATASET ──────────────────────────────────────────────────────

# define mosaic path again so it is independent of step 1 try/except scope
md = os.path.join(gdb, "RAW_GEBCO_MOSAIC_g4326")

# count rasters already in mosaic before adding
count_before = int(arcpy.management.GetCount(md)[0])


# Add Rasters to Mosaic Dataset (Data Management Tools):
try:
    arcpy.management.AddRastersToMosaicDataset(
        in_mosaic_dataset=md,
        raster_type="Raster Dataset",
        input_path=tifs,
        # "UPDATE_CELL_SIZES" (default) calculates cell size ranges for each input raster
        update_cellsize_ranges="UPDATE_CELL_SIZES",
        # "UPDATE_BOUNDARY" (default) creates/updates boundary polygon by merging all footprint polygons of input rasters into single boundary representing extent of valid pixels in mosaic
        update_boundary="UPDATE_BOUNDARY",
        # overview parameter applies to *mosaic* (overviews are lower-res images created to increase display speed/reduce CPU usage)
        update_overviews="NO_OVERVIEWS",
        # leave spatial_reference (defaults to coord system set for mosaic (same coord system as all input rasters) + sub_folder (default includes files from all subfolders within root input_path directory) blank
        # "OVERWRITE_DUPLICATES" new duplicate rasters overwrite existing rasters **matches on raster path so only overwrites if re-adding exact same file paths**
        duplicate_items_action="OVERWRITE_DUPLICATES",
        # pyramids parameter applies to *input rasters* (not mosaic itself); pyramid params default to resampling type "NEAREST" (ok here since passing raster data straight through, no reprojection/etc.); (pyramids are lower-res copy of data displayed when zoomed out and increases res as zoom in - termed "display resampling")
        build_pyramids="BUILD_PYRAMIDS",
        # "CALCULATE_STATISTICS" calculates stats for each input source raster (skips input rasters that already has stats IF skip factor value (default x=1 y=1) matches current settings)
        calculate_statistics="CALCULATE_STATISTICS"
    )
    print(f"✅ Added rasters to mosaic (see below messages for details!) \n")
# print warning if tool failed
except Exception as e:
    print(f"❌[FAILURE] Add rasters to mosaic failed: {e} \n")


# count rasters in mosaic after adding (*should be 12 since 12 input .tif files)
count_after = int(arcpy.management.GetCount(md)[0])
# newly added = net increase in raster count
added = count_after - count_before
# overwritten = inputs that were already registered (OVERWRITE_DUPLICATES replaces in place, so no net count change); infer from count math since this mosaic has no plain Path field to match against
overwritten = len(tifs) - added if added < len(tifs) else 0
print(f"🞥 Added new rasters = {added},    ⭮ Overwritten rasters = {overwritten},      🏱 Total rasters in mosaic = {count_after} \n")
# total n processed rasters should = n input files (*if processed < input it means some rasters failed to add - print warning message)
processed = added + overwritten
if processed < len(tifs):
    print(f"⚠️[WARNING] {len(tifs) - processed} raster(s) failed to add \n")
    print(arcpy.GetMessages())

✅ Added rasters to mosaic (see below messages for details!) 

🞥 Added new rasters = 12,    ⭮ Overwritten rasters = 0,      🏱 Total rasters in mosaic = 12 



In [21]:
# ── 3. EXPORT MOSAIC DATASET AS SINGLE STANDALONE RASTER ─────────────────────────────────────────────

# Copy Raster (Data Management Tools):
try:
    arcpy.management.CopyRaster(
        in_raster=md,
        out_rasterdataset=final_raster,
    )
    print(f"✅ Exported final raster: {final_raster} \n")

# print warning if export failed
except Exception as e:
    print(f"❌[FAILURE] Export raster failed: {e} \n")

✅ Exported final raster: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\GBC15a_bathy_entireEastCarib_withland_g4326 



In [22]:
# ── INSPECT OUTPUT RASTER DATA ───────────────────────────────────────────────────────────────────────

# Function to print all info (number of bands, pixel type, min, max, cell size, coordinate systems) for output raster
def print_raster_info(raster):

    # get number of bands info
    d = arcpy.Describe(raster)
    # get pixel type/min/max/cell size info (r = arcpy.Raster() loads raster dataset into a Raster object so arcpy can access stats)
    r = arcpy.Raster(raster)
    # get spatial reference info
    sr = d.spatialReference
    vcs = sr.VCS.name if sr.VCS else "NONE"

    # info print block for raster
    print(f"\n---------------- {os.path.basename(raster)} ----------------: \n")
    print(f"   Number bands = {d.bandCount}")
    print(f"   Pixel type = {r.pixelType}")
    print(f"   ⬇️ Min = {r.minimum}")
    print(f"   ⬆️ Max = {r.maximum}")
    print(f"   📐 Cell size = {r.meanCellWidth}")
    print(f"   Horizontal SR: {sr.name}")
    print(f"   Vertical SR: {vcs}")
    print(f"\n")


# Calculate Statistics (Data Management Tools): (default X/Y skip factor = 1)
arcpy.management.CalculateStatistics(in_raster_dataset=final_raster)

# Call function to print info for output raster
print_raster_info(final_raster)


---------------- GBC15a_bathy_entireEastCarib_withland_g4326 ----------------: 

   Number bands = 1
   Pixel type = S16
   ⬇️ Min = -8733.0
   ⬆️ Max = 5611.0
   📐 Cell size = 0.0041666666666666605
   Horizontal SR: GCS_WGS_1984
   Vertical SR: NONE




In [23]:
# ── 4. EXTRACT FOOTPRINT POLYGON FEATURE CLASS FROM MOSAIC ───────────────────────────────────────────

# Export Mosaic Dataset Geometry (Data Management Tools):
try:
    arcpy.management.ExportMosaicDatasetGeometry(
        in_mosaic_dataset=md, 
        out_feature_class=md_footprint, 
        geometry_type="FOOTPRINT"
    )
    print(f"✅ Created polygon feature class of mosaic footprint: {md_footprint} \n")

# print warning if tool failed
except Exception as e:
    print(f"❌[FAILURE] Footprint extraction failed: {e} \n")

✅ Created polygon feature class of mosaic footprint: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb\footprint_RAW_GEBCO_MOSAIC_g4326 

